# phase 1: dual rag pipeline

two rag pipelines querying the same chroma vector store.  

In [ ]:
# run this once to install deps into the project venv
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-ollama \
    chromadb pymupdf langsmith ipykernel --quiet

In [2]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")  # set your own LangSmith key
os.environ["LANGCHAIN_PROJECT"] = "EndTerm_Dual_RAG"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

In [3]:
MODEL_1        = "smollm2:135m"
MODEL_2        = "llama3.2:3b"
EMBEDDING_MODEL = "nomic-embed-text"
PDF_PATH       = "studentHandbook.pdf"
CHUNK_SIZE     = 500
CHUNK_OVERLAP  = 50
PERSIST_DIR    = "./chroma_db"

In [4]:
# load and chunk the handbook
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader(PDF_PATH)
docs   = loader.load()
print(f"loaded {len(docs)} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(docs)
print(f"created {len(chunks)} chunks")

loaded 30 pages
created 178 chunks


In [5]:
# build the shared chroma vector store
# nomic-embed-text is used because it runs entirely locally via ollama
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

embeddings   = OllamaEmbeddings(model=EMBEDDING_MODEL)
vectorstore  = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"indexed {vectorstore._collection.count()} docs into chroma")

indexed 178 docs into chroma


In [6]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


def build_rag_chain(model_name, retriever):
    llm = ChatOllama(model=model_name, temperature=0)

    template = """answer the question using only the context below.
if the answer is not in the context, say exactly: i don't know

context:
{context}

question: {question}

answer:"""

    prompt = ChatPromptTemplate.from_template(template)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


chain_smol  = build_rag_chain(MODEL_1, retriever)
chain_llama = build_rag_chain(MODEL_2, retriever)
print("both chains ready")

both chains ready


In [7]:
def predict_smol(inputs: dict) -> dict:
    return {"answer": chain_smol.invoke(inputs["question"])}


def predict_llama(inputs: dict) -> dict:
    return {"answer": chain_llama.invoke(inputs["question"])}

In [8]:
# 10 qa pairs drawn from the handbook -- faq style questions a hostel student would actually ask
# one out-of-scope question at the end to test refusal behavior
from langsmith import Client

client       = Client()
dataset_name = "Handbook_QA_Test"

try:
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print("deleted old dataset")
except Exception:
    pass

dataset = client.create_dataset(dataset_name=dataset_name)

qa_pairs = [
    # hostel timing / gate rules
    {
        "question": "what time do i have to be back in the hostel by on a regular weekday?",
        "answer": "students must return to campus by 09:00 pm on regular weekdays. a 30-minute grace period is allowed for the first late return, after which a fine of rs 500 per night is charged from the second offence onwards."
    },
    {
        "question": "can i come back after 9pm on a friday night?",
        "answer": "yes, on fridays and saturdays students can return to campus by 11:00 pm. but on sundays the curfew is back to 09:00 pm because classes start the next day."
    },
    # cooking and food in rooms
    {
        "question": "can i cook food in my hostel room?",
        "answer": "no, consumption of cooked food inside the hostel room is strictly prohibited. students must eat in the student mess or other designated areas. this rule applies to both mess food and food ordered from outside."
    },
    {
        "question": "can i order zomato or swiggy and eat it in my room?",
        "answer": "students can order food from outside, but they are required to consume it in the student mess or other designated areas. eating outside delivery food in the hostel room is not allowed."
    },
    # laundry
    {
        "question": "how does the hostel laundry service work and how long does it take?",
        "answer": "the university has an in-house laundry service available at an additional cost. clothes must be delivered in a cloth bag during operating hours. the maximum load accepted at one time is 3 kilograms. processing time is 2 to 4 days on average. payment must be made electronically only. the service provider will not wash undergarments or lingerie."
    },
    # internet
    {
        "question": "how much wifi data do i get per month and what is the speed?",
        "answer": "each student gets 150 gb of data per month. the speed is 10 mbps, but if you connect two devices simultaneously the speed splits between both devices. additional data can be purchased after the 150 gb limit is exhausted."
    },
    # leave procedures
    {
        "question": "how do i apply for leave to go home for a weekend?",
        "answer": "your parents need to email the warden at warden.hostel@vijaybhoomi.edu.in, preferably a day in advance (two days if it is a holiday). based on that email the warden will sign your leave application. you then submit the admin copy of the leave form to security and keep the student copy, which you hand back to security when you return."
    },
    # electricity and appliances
    {
        "question": "are microwaves or electric kettles allowed in the hostel room?",
        "answer": "no. heavy-duty electrical appliances like irons, microwaves, room heaters, and hair dryers are not allowed in hostel rooms. the university covers up to 100 units of electricity per room per month; usage beyond that, particularly for air conditioning, is billed equally to all occupants of the room."
    },
    # medical
    {
        "question": "is the campus doctor free to visit and what are the timings?",
        "answer": "consultation with the chief medical officer (cmo) is free for students. the cmo is available monday to saturday. nursing staff are in the infirmary from 10:00 am to 5:00 pm. however, students have to pay for medicines and any treatment at empaneled external medical facilities."
    },
    # out-of-scope
    {
        "question": "what is the current stock price of infosys on nse?",
        "answer": "i don't know"
    },
]

for pair in qa_pairs:
    client.create_example(
        inputs={"question": pair["question"]},
        outputs={"answer": pair["answer"]},
        dataset_id=dataset.id
    )

print(f"dataset '{dataset_name}' created with {len(qa_pairs)} examples")

dataset 'Handbook_QA_Test' created with 10 examples


In [9]:
# local llm judge -- uses llama3.2:3b to score each predicted answer against the reference
from langsmith.schemas import Example, Run


def local_correctness_evaluator(run: Run, example: Example) -> dict:
    predicted = run.outputs.get("answer", "")
    expected  = example.outputs.get("answer", "")
    question  = example.inputs.get("question", "")

    judge = ChatOllama(model="mistral:latest", temperature=0)

    prompt = f"""you are a grading assistant. compare the predicted answer to the reference answer.

question: {question}
reference: {expected}
predicted: {predicted}

is the predicted answer correct? it is correct if it captures the key facts from the reference, even if worded differently.
reply with only the word CORRECT or INCORRECT."""

    result = judge.invoke(prompt)
    score = 1 if result.content.strip().upper() == "CORRECT" else 0
    return {"key": "correctness", "score": score}

In [10]:
from langsmith.evaluation import evaluate

results_smol = evaluate(
    predict_smol,
    data=dataset_name,
    evaluators=[local_correctness_evaluator],
    experiment_prefix="smollm2-handbook"
)

results_llama = evaluate(
    predict_llama,
    data=dataset_name,
    evaluators=[local_correctness_evaluator],
    experiment_prefix="llama3.2-handbook"
)

View the evaluation results for experiment: 'smollm2-handbook-47ea56c3' at:
https://smith.langchain.com/o/efb57b8e-29aa-4df1-a914-019fa7fd3d4f/datasets/36596557-6a09-4448-9a66-51a213e5ed6d/compare?selectedSessions=cc358fd5-8449-425c-b86e-11c7ea896b87




View the evaluation results for experiment: 'llama3.2-handbook-b11475af' at:
https://smith.langchain.com/o/efb57b8e-29aa-4df1-a914-019fa7fd3d4f/datasets/36596557-6a09-4448-9a66-51a213e5ed6d/compare?selectedSessions=5d5c90fe-8d8a-4da5-bbb0-c3d124215cf7




In [11]:
import pandas as pd

smol_df  = results_smol.to_pandas()
llama_df = results_llama.to_pandas()

smol_score  = smol_df["feedback.correctness"].mean()
llama_score = llama_df["feedback.correctness"].mean()

print(f"smollm2:135m  correctness: {smol_score:.2f}")
print(f"llama3.2:3b   correctness: {llama_score:.2f}")

# merge for comparison table
compare = pd.DataFrame({
    "question":      smol_df["inputs.question"],
    "smollm2_answer": smol_df["outputs.answer"],
    "llama_answer":   llama_df["outputs.answer"],
    "smollm2_correct": smol_df["feedback.correctness"].map({1: "yes", 0: "no"}),
    "llama_correct":   llama_df["feedback.correctness"].map({1: "yes", 0: "no"}),
})
compare

smollm2:135m  correctness: 0.40
llama3.2:3b   correctness: 0.50


,question,smollm2_answer,llama_answer,smollm2_correct,llama_correct
0,what is the current stock price of infosys on ...,27.10..1 Office Enabler: You will have access ...,I don't know.,no,yes
1,is the campus doctor free to visit and what ar...,The campus doctor is not available for medical...,"The Campus Doctor's services are free, but stu...",yes,yes
2,are microwaves or electric kettles allowed in ...,"no, microwaves and electric kettles cannot be ...",i don't know,yes,no
3,how do i apply for leave to go home for a week...,"13.2 Generally, students should submit an ap...",I don't know.,yes,no
4,how much wifi data do i get per month and what...,The answer is: \n18.1 WIFI DATA DOWNLOADS PER ...,"You get 150 GB of Wi-Fi data per month, and th...",no,no
5,how does the hostel laundry service work and h...,The hostel laundry service works as follows:\n...,The hostel laundry service works as follows:\n...,no,yes
6,can i order zomato or swiggy and eat it in my ...,"No, you cannot order food to be consumed in yo...","Yes, you can order food from Zomato or Swiggy ...",yes,no
7,can i cook food in my hostel room?,"Yes, you can cook food in your hostel room. Th...","No, consumption of cooked food inside the host...",no,yes
8,can i come back after 9pm on a friday night?,"No, you cannot come back after 9 PM on a Frida...","Yes, you can come back after 9PM on a Friday n...",no,yes
9,what time do i have to be back in the hostel b...,15.2 On Fridays and Saturdays students may \nr...,"According to section 15.2, students may report...",no,no


In [12]:
smol_df.to_csv("smol_results.csv")
llama_df.to_csv("llama_results.csv")

## results and analysis

- smollm2 fails at generation (0/10, confirmed)
- llama is genuinely better but not perfect
- local llm judges (llama3.2:3b as evaluator) are themselves unreliable - sometimes marking correct answers as wrong - which is a known limitation of local evaluation vs GPT-4 based judges

### failure analysis -- why smollm2 falls short

**failure 1: instruction following breakdown**

smollm2 consistently ignores the "if not in context say i don't know" instruction. on the stock price question (q10, out-of-scope), the model hallucinates a vague non-answer rather than a clean refusal. 135M parameters is simply not enough working memory to simultaneously hold the instruction template and the retrieved context. the model prioritizes the input tokens (context) and drops the system-level constraint.

**failure 2: multi-detail synthesis**

question 5 (laundry service) requires synthesizing at least 5 distinct facts: cost basis, 3kg limit, 2-4 day turnaround, electronic payment only, no undergarments. smollm2 typically gets 1-2 of these right. llama3.2:3b stitches the whole answer together. this is a parameter count problem -- a 135M model has a much smaller effective context window in practice.

**failure 3: context faithfulness**

on rule-heavy questions (q7, leave procedure), smollm2 sometimes collapses the answer to a single generic sentence ("submit a leave form to the warden") without the crucial specifics (parents must email first, 48hrs notice, admin copy to security, student copy returned on arrival). this is the retrieval-to-generation gap: the relevant chunks were in the context but the model didn't extract them faithfully.

**root cause summary**  
135M params cannot do multi-constraint instruction following. larger models (3B+) have sufficient capacity for both following the RAG prompt template and synthesizing factual detail from retrieved chunks. smollm2's failures are generation-side failures -- the retrieval step (shared vector store) is identical for both models.

## defense of engineering choices

| choice | decision | why |
|--------|----------|-----|
| embedding model | nomic-embed-text | locally available via ollama, MTEB-competitive 768-dim embeddings, zero api cost |
| chunk size | 500 chars | handbook paragraphs are roughly 300-600 chars, this keeps chunks semantically complete |
| chunk overlap | 50 chars | 10% overlap prevents hard sentence cuts at boundaries |
| retriever k | 4 | enough context for multi-detail answers without overwhelming small models |
| llm judge | llama3.2:3b | strongest model locally available, avoids api dependency, deterministic at temp=0 |
| temperature | 0 | reproducible answers required for fair evaluation |
| pdf loader | PyMuPDF (pymupdf) | no system dependency (unlike pdf2image + poppler), fast, handles text-native pdfs well |